# 04 - Baseline Models

Kh?i ph?c baseline truy?n th?ng LR/KNN/DT/RF ?? ??i chi?u v?i anchor paper v? l?m m?c cho XGBoost/SMOTE.


In [1]:

## S6: Baseline models
# Run order: 01_eda.ipynb -> 02_feature_engineering.ipynb -> 03_prepare_dataset.ipynb -> 04_baseline_models.ipynb -> 05_extension_smote_xgboost.ipynb -> 06_shap_cost_sensitive.ipynb

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import time
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data'
results_dir = PROJECT_ROOT / 'results'

processed_path = DATA_DIR / 'processed_split.pkl'
results_dir.mkdir(exist_ok=True)

if not processed_path.exists():
    raise FileNotFoundError(
        'Missing ../data/processed_split.pkl. Run 03_prepare_dataset.ipynb first.'
    )

sys.path.append(str(PROJECT_ROOT / 'src'))
from evaluation import evaluate_model, print_metrics_table, compare_with_anchor_paper

# Load ??ng dict ?? l?u t? 03_prepare_dataset.ipynb.
data = joblib.load(processed_path)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']
X_train_scaled, X_test_scaled = data['X_train_scaled'], data['X_test_scaled']

print(f'Du lieu da load: X_train {X_train.shape}, X_test {X_test.shape}')
print(f'Train fraud rate: {y_train.mean():.4f}, Test fraud rate: {y_test.mean():.4f}')
print(f'So feature: {X_train.shape[1]}')


Du lieu da load: X_train (5090096, 15), X_test (1272524, 15)
Train fraud rate: 0.0013, Test fraud rate: 0.0013
So feature: 15


In [2]:

# Helper cho KNN: PaySim c? h?n 6.3 tri?u d?ng, KNN full train/full test r?t t?n th?i gian v? ph?i t?nh kho?ng c?ch.
# Notebook v?n ch?y ???c end-to-end b?ng stratified sample ri?ng cho KNN, c?n LR/DT/RF ??nh gi? tr?n full test set.
def stratified_positions(y, max_rows, random_state=42):
    y_arr = np.asarray(y)
    if len(y_arr) <= max_rows:
        return np.arange(len(y_arr))

    rng = np.random.default_rng(random_state)
    selected = []
    labels, counts = np.unique(y_arr, return_counts=True)
    remaining = max_rows

    for label, count in zip(labels, counts):
        label_positions = np.flatnonzero(y_arr == label)
        n_label = int(round(max_rows * count / len(y_arr)))
        if label == 1:
            n_label = max(n_label, min(count, 200))
        n_label = min(n_label, count, remaining)
        selected.append(rng.choice(label_positions, size=n_label, replace=False))
        remaining -= n_label

    if remaining > 0:
        already = np.concatenate(selected)
        pool = np.setdiff1d(np.arange(len(y_arr)), already, assume_unique=False)
        selected.append(rng.choice(pool, size=min(remaining, len(pool)), replace=False))

    positions = np.concatenate(selected)
    rng.shuffle(positions)
    return positions

baseline_results = []
baseline_models = {}


In [3]:

from sklearn.linear_model import LogisticRegression

start = time.time()
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_scaled, y_train)
lr_train_time = time.time() - start
print(f'Thoi gian train LR: {lr_train_time:.2f} giay')

lr_results = evaluate_model(lr, X_test_scaled, y_test, 'Logistic Regression', lr_train_time)
lr_results['evaluation_scope'] = 'full_test'
baseline_results.append(lr_results)
baseline_models['logistic_regression'] = lr
print(lr_results)


D:\GitHub\fraud-detection-thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Thoi gian train LR: 17.32 giay


{'model': 'Logistic Regression', 'accuracy': 0.9493329791815321, 'precision': 0.02392786785876648, 'recall': 0.9610468654899574, 'f1': 0.046693182322239145, 'roc_auc': 0.990774395026557, 'pr_auc': 0.5805680574507027, 'train_time_sec': 17.322, 'predict_time_sec': 0.169, 'evaluation_scope': 'full_test'}


In [4]:

from sklearn.neighbors import KNeighborsClassifier

KNN_TRAIN_MAX_ROWS = 50_000
KNN_TEST_MAX_ROWS = 50_000
knn_train_pos = stratified_positions(y_train, KNN_TRAIN_MAX_ROWS, random_state=42)
knn_test_pos = stratified_positions(y_test, KNN_TEST_MAX_ROWS, random_state=43)

X_train_knn = X_train_scaled[knn_train_pos]
y_train_knn = y_train.iloc[knn_train_pos]
X_test_knn = X_test_scaled[knn_test_pos]
y_test_knn = y_test.iloc[knn_test_pos]

print(f'KNN train sample: {X_train_knn.shape}, fraud rate={y_train_knn.mean():.4f}')
print(f'KNN test sample: {X_test_knn.shape}, fraud rate={y_test_knn.mean():.4f}')

start = time.time()
knn = KNeighborsClassifier(n_neighbors=5, algorithm='auto', n_jobs=-1)
knn.fit(X_train_knn, y_train_knn)
knn_train_time = time.time() - start
print(f'Thoi gian train KNN: {knn_train_time:.2f} giay')

knn_results = evaluate_model(knn, X_test_knn, y_test_knn, 'KNN', knn_train_time)
knn_results['evaluation_scope'] = f'stratified_sample_train_{len(knn_train_pos)}_test_{len(knn_test_pos)}'
baseline_results.append(knn_results)
baseline_models['knn'] = knn
print(knn_results)


KNN train sample: (50000, 15), fraud rate=0.0013
KNN test sample: (50000, 15), fraud rate=0.0013


Thoi gian train KNN: 0.23 giay


{'model': 'KNN', 'accuracy': 0.99912, 'precision': 1.0, 'recall': 0.3230769230769231, 'f1': 0.4883720930232558, 'roc_auc': 0.8073198234628094, 'pr_auc': 0.49064370245139477, 'train_time_sec': 0.234, 'predict_time_sec': 9.479, 'evaluation_scope': 'stratified_sample_train_50000_test_50000'}


In [5]:

from sklearn.tree import DecisionTreeClassifier

start = time.time()
dt = DecisionTreeClassifier(random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
dt_train_time = time.time() - start
print(f'Thoi gian train Decision Tree: {dt_train_time:.2f} giay')

dt_results = evaluate_model(dt, X_test, y_test, 'Decision Tree', dt_train_time)
dt_results['evaluation_scope'] = 'full_test'
baseline_results.append(dt_results)
baseline_models['decision_tree'] = dt
print(dt_results)


Thoi gian train Decision Tree: 150.97 giay


{'model': 'Decision Tree', 'accuracy': 0.9999921416020444, 'precision': 0.9963525835866262, 'recall': 0.9975654290931223, 'f1': 0.9969586374695864, 'roc_auc': 0.9987803539793639, 'pr_auc': 0.993930035932816, 'train_time_sec': 150.973, 'predict_time_sec': 0.366, 'evaluation_scope': 'full_test'}


In [6]:

from sklearn.ensemble import RandomForestClassifier

start = time.time()
rf = RandomForestClassifier(
    n_estimators=50,
    random_state=42,
    class_weight='balanced_subsample',
    n_jobs=-1,
    min_samples_leaf=2
)
rf.fit(X_train, y_train)
rf_train_time = time.time() - start
print(f'Thoi gian train Random Forest: {rf_train_time:.2f} giay')

rf_results = evaluate_model(rf, X_test, y_test, 'Random Forest', rf_train_time)
rf_results['evaluation_scope'] = 'full_test'
baseline_results.append(rf_results)
baseline_models['random_forest'] = rf
print(rf_results)


Thoi gian train Random Forest: 352.24 giay


{'model': 'Random Forest', 'accuracy': 0.9999968566408177, 'precision': 1.0, 'recall': 0.9975654290931223, 'f1': 0.9987812309567337, 'roc_auc': 0.9996571940639433, 'pr_auc': 0.9982008720319745, 'train_time_sec': 352.238, 'predict_time_sec': 5.027, 'evaluation_scope': 'full_test'}


In [7]:

baseline_comparison = print_metrics_table(baseline_results)
print('Bang baseline LR/KNN/DT/RF:')
print(baseline_comparison)

comparison = compare_with_anchor_paper(baseline_comparison)
print('\nSo sanh voi anchor paper:')
print(comparison)

baseline_comparison.to_csv(results_dir / 'baseline_results.csv', index=False)
joblib.dump(baseline_models, results_dir / 'baseline_models.pkl')
print('\nDa luu baseline results vao ../results/baseline_results.csv')
print('Da luu baseline models vao ../results/baseline_models.pkl')


Bang baseline LR/KNN/DT/RF:
                 model  accuracy  precision  recall      f1  roc_auc  pr_auc  \
0  Logistic Regression    0.9493     0.0239  0.9610  0.0467   0.9908  0.5806   
1                  KNN    0.9991     1.0000  0.3231  0.4884   0.8073  0.4906   
2        Decision Tree    1.0000     0.9964  0.9976  0.9970   0.9988  0.9939   
3        Random Forest    1.0000     1.0000  0.9976  0.9988   0.9997  0.9982   

   train_time_sec  predict_time_sec                          evaluation_scope  
0          17.322             0.169                                 full_test  
1           0.234             9.479  stratified_sample_train_50000_test_50000  
2         150.973             0.366                                 full_test  
3         352.238             5.027                                 full_test  



So sanh voi anchor paper:
                 model  accuracy_yours  accuracy_paper  precision_yours  \
0  Logistic Regression          0.9493          0.9705           0.0239   
1                  KNN          0.9991          0.9954           1.0000   
2        Decision Tree          1.0000          0.9994           0.9964   
3        Random Forest          1.0000          0.9932           1.0000   

   precision_paper  recall_yours  recall_paper  f1_yours  f1_paper  \
0           0.0382        0.9610        0.9018    0.0467    0.0732   
1           0.1952        0.3231        0.8255    0.4884    0.3157   
2           0.6835        0.9976        0.9696    0.9970    0.8018   
3           0.1576        0.9976        0.9838    0.9988    0.2716   

   roc_auc_yours  roc_auc_paper  
0         0.9908         0.9838  
1         0.8073         0.9186  
2         0.9988         0.9845  
3         0.9997         0.9990  



Da luu baseline results vao ../results/baseline_results.csv
Da luu baseline models vao ../results/baseline_models.pkl
